# basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import numpy as np
import pickle
import itertools

import src.configs as configs
from src.tokenizer import GraphTokenizer
from src.iterative_selection import iterative_approach_w_graph_red

# 0. load files

In [3]:
with open(configs.ProcessedGraph().candidate_is_a_reachable_dict, "rb") as f:
    candidate_reachable_child_map = pickle.load(f)

df_concept_hug_w_depth = pl.read_parquet(configs.ProcessedGraph.concept_w_depth)
mapped_candidate_rel_dist_prop = (pl.read_parquet(configs.ProcessedGraph().mapped_candidate_rel_dist_prop)
                                  
                                  # remove all relations in subgraph where candidate and mapped are distant
                                  .filter(pl.col("distance") <= configs.TokenizerParam().max_dist_candidate)
                                  )

# 1. iterative with graph reduction

In [4]:
distance_score_types = ["tempered_sum"]# ["sum", "tempered_sum"]
distance_type = ["inv_exp"]# ["inv", "inv_exp"]

for sc_type, d_type in itertools.product(distance_score_types, distance_type):
    candidate_rows = iterative_approach_w_graph_red(
        mapped_candidate_rel_dist_prop,
        candidate_reachable_child_map,
        sc_type,
        d_type,
    )
    pl.DataFrame(candidate_rows).write_parquet(f"{configs.IterativeGraphRed().path} {sc_type}_{d_type}.parquet")
    print(f"{sc_type} / {d_type}: {len(candidate_rows)} candidates selected")

number of candidates selected: 500, number of rows remaining: 417221
number of candidates selected: 1000, number of rows remaining: 329179
number of candidates selected: 1500, number of rows remaining: 288759
number of candidates selected: 2000, number of rows remaining: 273790
number of candidates selected: 2500, number of rows remaining: 273290
number of candidates selected: 3000, number of rows remaining: 272790
number of candidates selected: 3500, number of rows remaining: 272290
number of candidates selected: 4000, number of rows remaining: 271790
number of candidates selected: 4500, number of rows remaining: 271290
number of candidates selected: 5000, number of rows remaining: 270790
number of candidates selected: 5500, number of rows remaining: 270290
number of candidates selected: 6000, number of rows remaining: 269790
number of candidates selected: 6500, number of rows remaining: 269290
number of candidates selected: 7000, number of rows remaining: 268790
number of candidates 

# tokenizer test (max dist  == 3)

In [5]:
tokenizer = GraphTokenizer(df_concept_hug_w_depth,
                        mapped_candidate_rel_dist_prop,
                        candidate_reachable_child_map,
                        configs.TokenizerParam().max_dist_candidate
                        )

## tryout

In [6]:
# top 8000 candidates of highest degree
for k in np.arange(1000, 10000, 1000):
    example_candidates = mapped_candidate_rel_dist_prop.group_by("dst.id").agg(num_src = pl.col("src.id").n_unique()).sort(by = "num_src", descending=True).head(k)["dst.id"].to_list()
    scores, results, df_tok_all_n_dist = tokenizer.evaluate_components_and_tokenize(example_candidates, debug=True)


In [7]:
df_tok_all_n_dist# ["candidate_id"].n_unique()

mapped_id,candidate_id,relation,distance
str,str,str,f64
"""73862001""","""73862001""","""IS_A""",0.0
"""254502002""","""254502002""","""IS_A""",0.0
"""127337006""","""127337006""","""IS_A""",0.0
"""414403008""","""414403008""","""IS_A""",0.0
"""88425004""","""88425004""","""IS_A""",0.0
…,…,…,…
"""277387000""","""UNK""","""IS_A""",null
"""1296723006""","""UNK""","""IS_A""",null
"""10083006""","""UNK""","""IS_A""",null
